# Day 5: SEC EDGAR 10-K テキスト取得テスト

**目的**: 全年度の10-Kを一括取得してNLP前処理パイプラインを構築する工数を見積もる

**判断ポイント**: フォーマットの年代差がどの程度あるか → 前処理の分岐ロジック量

In [ ]:
import requests
import json
import re
import os
from bs4 import BeautifulSoup
import plotly.graph_objects as go

## 1. 10-K Filing一覧

In [ ]:
CIK = "0000829224"
headers = {"User-Agent": "seto seto.biz@icloud.com"}

# Recent filings
url = f"https://data.sec.gov/submissions/CIK{CIK}.json"
r = requests.get(url, headers=headers, timeout=30)
data = r.json()
recent = data['filings']['recent']

tenk_recent = []
for i, form in enumerate(recent['form']):
    if form == '10-K':
        tenk_recent.append({
            'date': recent['filingDate'][i],
            'accession': recent['accessionNumber'][i],
            'primaryDoc': recent['primaryDocument'][i],
        })

# Archive filings
url2 = f"https://data.sec.gov/submissions/CIK{CIK}-submissions-001.json"
r2 = requests.get(url2, headers=headers, timeout=30)
arch = r2.json()

tenk_archive = []
for i, form in enumerate(arch['form']):
    if form == '10-K':
        tenk_archive.append({
            'date': arch['filingDate'][i],
            'accession': arch['accessionNumber'][i],
            'primaryDoc': arch['primaryDocument'][i],
        })

all_tenk = tenk_recent + tenk_archive
print(f"10-K filing総数: {len(all_tenk)}")
print(f"\n{'日付':12s} | {'形式':5s} | {'accession'}")
print('-' * 60)
for tk in all_tenk:
    doc = tk['primaryDoc']
    fmt = 'HTM' if doc.endswith('.htm') else 'TXT' if doc.endswith('.txt') else '不明' if not doc else doc[-4:]
    print(f"{tk['date']:12s} | {fmt:5s} | {tk['accession']}")

## 2. テキスト抽出テスト (3ファイル比較)

In [ ]:
def extract_text(filepath):
    """HTML/TXT両対応のテキスト抽出"""
    with open(filepath, 'r', encoding='utf-8', errors='replace') as f:
        raw = f.read()
    if filepath.endswith('.htm') or filepath.endswith('.html'):
        soup = BeautifulSoup(raw, 'html.parser')
        return soup.get_text(separator='\n', strip=True)
    return raw

def find_sections(text):
    """10-Kのセクション開始位置を検出（TOC除外）"""
    sections = {}
    patterns = [
        ('Item 1 Business', r'(?i)\bITEM\s+1[\.:\s\u2014\u2013-]+\s*BUSINESS\b'),
        ('Item 1A Risk Factors', r'(?i)\bITEM\s+1A[\.:\s\u2014\u2013-]+\s*RISK\s+FACTORS\b'),
        ('Item 2 Properties', r'(?i)\bITEM\s+2[\.:\s\u2014\u2013-]+\s*PROPERTIES\b'),
        ('Item 7 MD&A', r'(?i)\bITEM\s+7[\.:\s\u2014\u2013-]+\s*MANAGEMENT'),
        ('Item 8 Financial', r'(?i)\bITEM\s+8[\.:\s\u2014\u2013-]+\s*FINANCIAL'),
    ]
    for name, pattern in patterns:
        for m in re.finditer(pattern, text):
            after = text[m.end():m.end()+200]
            if re.search(r'(?i)\bITEM\s+\d', after) and re.search(r'(?i)\bITEM\s+\d', after).start() < 100:
                continue
            sections[name] = m.start()
            break
    return sections

base = "../../data/raw/sec_10k"
files = {'FY2025': '10k_fy2025.htm', 'FY2001': '10k_fy2001.htm', 'FY2000': '10k_fy2000.txt'}

for label, fname in files.items():
    text = extract_text(os.path.join(base, fname))
    sections = find_sections(text)
    sorted_sec = sorted(sections.items(), key=lambda x: x[1])
    
    print(f"\n{'='*50}")
    print(f"{label} ({fname}) — {len(text):,}文字")
    for i, (name, pos) in enumerate(sorted_sec):
        end = sorted_sec[i+1][1] if i+1 < len(sorted_sec) else len(text)
        print(f"  {name:25s}: {end-pos:,}文字")
    
    dollar_count = text.count('$')
    print(f"  $記号(数表混入指標): {dollar_count}")

## 3. Item 1 Business テキスト品質確認

In [ ]:
# 最新と最古のItem 1を比較表示
for label, fname in [('FY2025', '10k_fy2025.htm'), ('FY2000', '10k_fy2000.txt')]:
    text = extract_text(os.path.join(base, fname))
    sections = find_sections(text)
    if 'Item 1 Business' in sections:
        start = sections['Item 1 Business']
        # 次のセクションまで
        sorted_sec = sorted(sections.items(), key=lambda x: x[1])
        idx = [i for i, (n, _) in enumerate(sorted_sec) if n == 'Item 1 Business'][0]
        end = sorted_sec[idx+1][1] if idx+1 < len(sorted_sec) else start + 5000
        snippet = text[start:min(start+1000, end)]
        print(f"\n{'='*50}")
        print(f"{label} Item 1 Business (先頭1000文字)")
        print(f"{'='*50}")
        print(snippet)

In [ ]:
# テキスト量の年代比較
data_for_plot = {
    'FY2025': {'Item 1': 37965, 'Item 7 MD&A': 222393, 'format': 'XBRL HTM'},
    'FY2001': {'Item 1': 21570, 'Item 7 MD&A': 789, 'format': 'HTML'},
    'FY2000': {'Item 1': 19789, 'Item 7 MD&A': 697, 'format': 'TXT'},
}

fig = go.Figure()
years = list(data_for_plot.keys())
fig.add_trace(go.Bar(name='Item 1 Business', x=years,
                     y=[d['Item 1'] for d in data_for_plot.values()],
                     marker_color='#00704A'))
fig.add_trace(go.Bar(name='Item 7 MD&A', x=years,
                     y=[d['Item 7 MD&A'] for d in data_for_plot.values()],
                     marker_color='#D4A76A'))
fig.update_layout(title='10-K セクション別テキスト量',
                  yaxis_title='文字数', barmode='group',
                  template='plotly_white', width=700, height=400)
fig.show()

print("注意: FY2000-2001のItem 7は'incorporated by reference'のため本文なし")

## 4. 判定結果

### フォーマット分岐: 3パターン

| 年代 | 形式 | サイズ | 特徴 |
|---|---|---|---|
| 2012-2025 | XBRL HTM | ~3MB | iXBRL埋込。メタデータ除去必要 |
| 2001-2011 | HTML | ~100KB | シンプルHTML |
| 2000以前 | TXT | ~56KB | プレーンテキスト。OCR不要 |
| 1996-1999 | 不明 | 取得困難 | primaryDoc空 / 503エラー |

### 重要な発見

1. **古い10-K (2000-2011) の Item 7 MD&A は "incorporated by reference"**
   - 10-K単体ではMD&A本文がない
   - Annual Report別添付を取得する必要あり（高工数）

2. **Item 1 Business は全年度で十分なテキスト量** (20K-40K文字/年)
   - NLPのメインターゲットはItem 1にすべき

3. **セクション分割ロジック**: TOC除外付きで3パターン対応 → 中程度の工数

4. **テキスト品質**: 文字化けなし、NLPに十分な品質

### 工数影響

- 前処理パイプライン: 3パターン分岐のみ → **管理可能**
- Item 1をメインにすれば "incorporated by reference" 問題を回避できる
- 1996-1999年は手動取得が必要だが4ファイルのみ